In [20]:
# pip install arch==8.0.0

In [21]:
import numpy as np
import pandas as pd
from arch import arch_model 

# Phase 6: GARCH(1,1) Volatility Scaled Historical Simulation VaR & ES

In [22]:
# Step 1: Loading Phase 2 Outputs
portfolio_dollar_value = pd.read_csv(r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\portfolio_dollar_value.csv', index_col =0, parse_dates =True).iloc[:,0]
portfolio_pnl = pd.read_csv(r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\portfolio_pnl.csv', index_col =0, parse_dates =True).iloc[:,0]

In [23]:
port_simple_returns = portfolio_dollar_value.pct_change().dropna()

In [24]:
port_log_returns = np.log(portfolio_dollar_value / portfolio_dollar_value.shift(1)).dropna()

In [25]:
log_returns_pct = port_log_returns*100

In [26]:
# Step 2: Fit GARCH(1,1) once on an initial training window only

In [27]:
TRAIN_WINDOW = 750
garch_train_returns = log_returns_pct.iloc[:TRAIN_WINDOW]
garch_spec = arch_model(garch_train_returns, vol = 'GARCH', p = 1, q = 1, dist ='normal', mean = 'constant')
garch_fit = garch_spec.fit(disp = 'off')

mu = garch_fit.params['mu']
omega = garch_fit.params['omega']
alpha = garch_fit.params['alpha[1]']
beta = garch_fit.params['beta[1]']

print(f"\nFitted GARCH(1,1) parameters (frozen for the whole backtest): ")
print(f" mu = {mu: .6f}  omega = {omega: .6f}  alpha = {alpha: .4f}  beta = {beta: .4f}  (alpha + beta = {alpha + beta: .4f})")

assert alpha + beta < 1, (
    f"Non stationary GARCH fit (alpha + beta = { alpha + beta: .4f} >= 1) - "
    f"the model has no mean reverting long run variance."
)


Fitted GARCH(1,1) parameters (frozen for the whole backtest): 
 mu =  0.096594  omega =  0.133412  alpha =  0.1678  beta =  0.7502  (alpha + beta =  0.9181)


In [28]:
# Step 3: Building the full conditional volatility series

In [29]:
n = len(log_returns_pct)
garch_variance_pct = np.full(n, np.nan)
garch_variance_pct[: TRAIN_WINDOW] = garch_fit.conditional_volatility.values**2

log_returns_pct_array = log_returns_pct.values
for t in range(TRAIN_WINDOW, n):
    resid_prev = log_returns_pct_array[t-1] - mu
    garch_variance_pct[t] = omega + alpha * (resid_prev**2) + beta * garch_variance_pct[t-1]

garch_vol = pd.Series(np.sqrt(garch_variance_pct) / 100, index = port_log_returns.index)

standardized_residuals = (port_log_returns - mu / 100) / garch_vol
standardized_residuals.name = 'GARCH_Standardized_Residuals'

In [30]:
# Step 4: Aligning to a common date index

In [31]:
common_idx = port_simple_returns.index.intersection(garch_vol.index)
 
assert len(common_idx) > 0, (
    "No overlapping dates between port_simple_returns and garch_vol. "
)
 
port_simple_returns_aligned = port_simple_returns.loc[common_idx]
garch_vol_aligned = garch_vol.loc[common_idx]
 
print(f"\nTotal available trading days for GARCH engine: {len(port_simple_returns_aligned)}")


Total available trading days for GARCH engine: 1506


In [32]:
# Step 5: Parameters & Core Risk Function (reused from Phase 4/5)

In [33]:
windows = [250, 500, 750]
confidence_levels = [0.95, 0.99, 0.995]
MAX_SCALING_RATIO = 3.0
 
def calculate_historical_var_es(returns_window, conf_level):
    percentile = (1 - conf_level) * 100
    var_pct = -np.percentile(returns_window, percentile, method='lower')
    tail_returns = returns_window[returns_window <= -var_pct]
    es_pct = -tail_returns.mean() if len(tail_returns) > 0 else var_pct
    return var_pct, es_pct

In [34]:
# Step 6: Initializing the Rolling Engine 

In [35]:
garch_metrics = pd.DataFrame(index=port_simple_returns_aligned.index)
garch_metrics['Portfolio_Dollar_Value'] = portfolio_dollar_value.loc[port_simple_returns_aligned.index]
garch_metrics['Daily_PnL'] = portfolio_pnl.loc[port_simple_returns_aligned.index]
garch_metrics['Portfolio_Dollar_Value_Prior'] = garch_metrics['Portfolio_Dollar_Value'].shift(1)
garch_metrics['GARCH_Vol_Daily'] = garch_vol_aligned
garch_metrics['GARCH_Standardized_Residuals'] = standardized_residuals.loc[port_simple_returns_aligned.index]
 
ret_array = port_simple_returns_aligned.values
vol_array = garch_vol_aligned.values

In [36]:
# Step 7: Executing Rolling Volatility Scaled Calculations

In [37]:
for w in windows:
    for cl in confidence_levels:
        print(f"Calculating GARCH-Scaled {w}-day window at {cl * 100}% confidence...")
 
        var_list = np.full(len(ret_array), np.nan)
        es_list = np.full(len(ret_array), np.nan)
 
        for i in range(w, len(ret_array)):
            window_returns = ret_array[i - w:i]
            window_vols = vol_array[i - w:i]
            current_vol_forecast = vol_array[i]
 
            scaling_ratios = current_vol_forecast / window_vols
            scaling_ratios = np.clip(scaling_ratios, a_min=None, a_max=MAX_SCALING_RATIO)
            scaled_window_returns = window_returns * scaling_ratios
 
            v, e = calculate_historical_var_es(scaled_window_returns, cl)
            var_list[i] = v
            es_list[i] = e
 
        col_base = f"{w}d_{cl * 100}%"
        garch_metrics[f"GARCH_VaR_pct_{col_base}"] = var_list
        garch_metrics[f"GARCH_ES_pct_{col_base}"] = es_list
 
        garch_metrics[f"GARCH_VaR_1d_$_" + col_base] = (
            garch_metrics[f"GARCH_VaR_pct_{col_base}"] * garch_metrics['Portfolio_Dollar_Value_Prior']
        )
        garch_metrics[f"GARCH_ES_1d_$_" + col_base] = (
            garch_metrics[f"GARCH_ES_pct_{col_base}"] * garch_metrics['Portfolio_Dollar_Value_Prior']
        )
        garch_metrics[f"GARCH_VaR_10d_$_" + col_base] = garch_metrics[f"GARCH_VaR_1d_$_" + col_base] * np.sqrt(10)
        garch_metrics[f"GARCH_ES_10d_$_" + col_base] = garch_metrics[f"GARCH_ES_1d_$_" + col_base] * np.sqrt(10)

Calculating GARCH-Scaled 250-day window at 95.0% confidence...
Calculating GARCH-Scaled 250-day window at 99.0% confidence...
Calculating GARCH-Scaled 250-day window at 99.5% confidence...
Calculating GARCH-Scaled 500-day window at 95.0% confidence...
Calculating GARCH-Scaled 500-day window at 99.0% confidence...
Calculating GARCH-Scaled 500-day window at 99.5% confidence...
Calculating GARCH-Scaled 750-day window at 95.0% confidence...
Calculating GARCH-Scaled 750-day window at 99.0% confidence...
Calculating GARCH-Scaled 750-day window at 99.5% confidence...


In [38]:
# Step 8: Finalizing the out of sample dataset

In [39]:
out_of_sample_garch = garch_metrics.dropna(subset=['GARCH_VaR_1d_$_750d_99.0%'])
 
stray_nan_count = out_of_sample_garch.isna().sum().sum()
if stray_nan_count > 0:
    print(f"WARNING: {stray_nan_count} unexpected NaN value(s) remain outside the checked column.")
else:
    print("No stray NaN values found elsewhere in the out-of-sample dataset.")
 
print(f"\nPhase 6 Complete. Out-of-sample backtesting days available: {len(out_of_sample_garch)}")

No stray NaN values found elsewhere in the out-of-sample dataset.

Phase 6 Complete. Out-of-sample backtesting days available: 756


In [40]:
# Step 9: Validations (checked across all out of sample days)

In [41]:
var_1d = out_of_sample_garch['GARCH_VaR_1d_$_750d_99.0%']
var_10d = out_of_sample_garch['GARCH_VaR_10d_$_750d_99.0%']
es_1d = out_of_sample_garch['GARCH_ES_1d_$_750d_99.0%']
 
assert (var_1d > 0).all(), "Validation Failed: GARCH VaR must be positive on every day."
assert (var_10d > var_1d).all(), "Validation Failed: 10-day GARCH VaR must exceed 1-day on every day."
assert (es_1d >= var_1d).all(), "Validation Failed: GARCH ES must be >= GARCH VaR on every day."
 
var_95 = out_of_sample_garch['GARCH_VaR_1d_$_750d_95.0%']
var_99 = out_of_sample_garch['GARCH_VaR_1d_$_750d_99.0%']
var_995 = out_of_sample_garch['GARCH_VaR_1d_$_750d_99.5%']
 
assert (var_99 >= var_95).all(), "Validation Failed: 99% GARCH VaR must be >= 95% on every day."
assert (var_995 >= var_99).all(), "Validation Failed: 99.5% GARCH VaR must be >= 99% on every day."
 
print("All mathematical constraints validated across the full out-of-sample period.")

All mathematical constraints validated across the full out-of-sample period.


In [42]:
# Step 10: Saving results for dashboarding, Phase 8 and Phase 9 comparison

In [43]:
out_of_sample_garch.to_csv('garch_scaled_var_es.csv')
print("Saved: garch_scaled_var_es.csv")

Saved: garch_scaled_var_es.csv
